In [1]:
# TITULO: Entrenamiento Comparativo - Detector de Placas
import os
from ultralytics import YOLO
from datetime import datetime
import pandas as pd
import matplotlib.pyplot as plt

# Rutas de los datasets generados
YAML_CLEAN = '../../datasets/02_placas/data.yaml'
YAML_BASELINE = '../../datasets/02_placas_baseline/data.yaml'

# Ruta de salida de modelos
MODELS_DIR = '../../models/02_placas'

# Fecha para versionado
DATE_STR = datetime.now().strftime('%Y%m%d')

print("Configuracion lista.")

Configuracion lista.


In [17]:
# Nombre del experimento
run_name_clean = f"{DATE_STR}_v8n_clean_tl_640"

print(f"Iniciando entrenamiento: {run_name_clean}")

# Cargar modelo Nano pre-entrenado
model_clean = YOLO('yolov8n.pt')

results_clean = model_clean.train(
    data=YAML_CLEAN,
    project=MODELS_DIR,
    name=run_name_clean,
    epochs=30,            # Ajustable
    imgsz=640,
    batch=-1,          # Ajustable
    patience=10,          # Early stopping
    exist_ok=True,         # Sobrescribir si existe
    verbose=True
)

Iniciando entrenamiento: 20251201_v8n_clean_tl_640
Ultralytics 8.3.233 🚀 Python-3.11.14 torch-2.9.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4070 Ti, 11852MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../../datasets/02_placas/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=20251201_v8n_clean_tl_640, nbs=64, nms=False, opset=None, optimi

In [2]:
# Nombre del experimento
run_name_base = f"{DATE_STR}_v8n_baseline_tl_640"

print(f"Iniciando entrenamiento Baseline: {run_name_base}")

# Cargar modelo nuevo para no contaminar pesos
model_base = YOLO('yolov8n.pt')

results_base = model_base.train(
    data=YAML_BASELINE,
    project=MODELS_DIR,
    name=run_name_base,
    epochs=30,
    imgsz=640,
    batch=-1,
    patience=10,
    exist_ok=True,
    verbose=True
)

Iniciando entrenamiento Baseline: 20251201_v8n_baseline_tl_640
Ultralytics 8.3.233 🚀 Python-3.11.14 torch-2.9.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4070 Ti, 11852MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../../datasets/02_placas_baseline/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=20251201_v8n_baseline_tl_640, nbs=64, nms=F

In [3]:
# Cargar mejores pesos
# Asegúrate que estos nombres coincidan exactamente con los definidos en las celdas de entrenamiento
run_name_clean = f"{DATE_STR}_v8n_clean_tl_640"
run_name_base = f"{DATE_STR}_v8n_baseline_tl_640" # Nota: Ajusta si usaste otro nombre en la celda 3

path_clean_weights = os.path.join(MODELS_DIR, run_name_clean, 'weights', 'best.pt')
path_base_weights = os.path.join(MODELS_DIR, run_name_base, 'weights', 'best.pt')

model_final_clean = YOLO(path_clean_weights)
model_final_base = YOLO(path_base_weights)

print("--- VALIDACION CRUZADA ---")

# 1. Validar Modelo Propuesto (Clean)
# Validamos contra el set de datos limpio (test) y definimos nombre específico
metrics_clean = model_final_clean.val(
    data=YAML_CLEAN, 
    split='test', 
    project=MODELS_DIR,
    name=f"{run_name_clean}_val"  # <--- CAMBIO: Nombre personalizado para la carpeta
)

# 2. Validar Modelo Original (Baseline)
# Usamos el MISMO set de datos limpio como 'Gold Standard' para comparar justamente
metrics_base = model_final_base.val(
    data=YAML_CLEAN, 
    split='test', 
    project=MODELS_DIR,
    name=f"{run_name_base}_val"   # <--- CAMBIO: Nombre personalizado para la carpeta
)

print("\nRESULTADOS COMPARATIVOS (mAP50-95):")
print(f"Modelo Propuesto (Clean Split): {metrics_clean.box.map:.4f}")
print(f"Modelo Original (Raw Split):    {metrics_base.box.map:.4f}")

--- VALIDACION CRUZADA ---
Ultralytics 8.3.233 🚀 Python-3.11.14 torch-2.9.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4070 Ti, 11852MiB)
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 689.2±41.7 MB/s, size: 2479.1 KB)
val: Scanning /home/roberto/moca_proyecto/datasets/02_placas/test/labels... 198 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 198/198 3.2Kit/s 0.1s
val: New cache created: /home/roberto/moca_proyecto/datasets/02_placas/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 13/13 1.5it/s 8.9s0.3s
                   all        198        242      0.977      0.942      0.985      0.848
Speed: 1.1ms preprocess, 1.8ms inference, 0.0ms loss, 1.1ms postprocess per image
Results saved to /home/roberto/moca_proyecto/models/02_placas/20251201_v8n_clean_tl_640_val
Ultralytics 8.3.233 🚀 Python-3.11.14 torch-2.9.1+cu126 CUDA:0 (NVIDIA G

In [4]:
import glob
import random

run_name_clean = f"{DATE_STR}_v8_clean_tl_640_inference"

# Tomar una imagen de prueba aleatoria
test_images = glob.glob('../../datasets/02_placas/test/images/*.jpg')
if test_images:
    sample_img = random.choice(test_images)
    
    # Prediccion
    res = model_final_clean.predict(sample_img, save=True, project=MODELS_DIR, name=run_name_clean)
     
    print(f"Inferencia guardada en {res[0].save_dir}")
else:
    print("No se encontraron imagenes de prueba.")


image 1/1 /home/roberto/moca_proyecto/notebooks/02_placas/../../datasets/02_placas/test/images/00183.jpg: 480x640 1 license_plate, 29.1ms
Speed: 2.3ms preprocess, 29.1ms inference, 1.1ms postprocess per image at shape (1, 3, 480, 640)
Results saved to /home/roberto/moca_proyecto/models/02_placas/20251201_v8_clean_tl_640_inference
Inferencia guardada en /home/roberto/moca_proyecto/models/02_placas/20251201_v8_clean_tl_640_inference
